# R के साथ सांख्यिकी

R के अंतर्निहित डेटासेट का उपयोग करके परिकल्पना परीक्षण, प्रतिगमन (regression), और विश्वास अंतराल (confidence intervals)।

किसी डेटा डाउनलोड या पैकेज स्थापना की आवश्यकता नहीं है — केवल बेस R का उपयोग करता है।

## 1. विवरणात्मक सांख्यिकी

In [ ]:
data(mtcars)
cat("डेटासेट: mtcars (", nrow(mtcars), " कारें, ", ncol(mtcars), " चर)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. दो-नमूना टी-परीक्षण (Two-Sample t-Test)

क्या मैनुअल ट्रांसमिशन वाली कारों का ईंधन माइलेज (MPG) स्वचालित कारों की तुलना में बेहतर है?

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("स्वचालित:", round(mean(auto), 1), "MPG (n =", length(auto), ")\n")
cat("मैन्युअल:   ", round(mean(manual), 1), "MPG (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nनिष्कर्ष:",
    ifelse(t_result$p.value < 0.05,
           "H0 को अस्वीकार करें — मैनुअल कारों का MPG काफी अधिक है",
           "H0 को अस्वीकार करने में विफल"))

## 3. काई-स्क्वायर परीक्षण (Chi-Square Test)

क्या सिलेंडर की संख्या और ट्रांसमिशन का प्रकार स्वतंत्र हैं?

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("स्वचालित", "मैन्युअल")
print(tab)
cat("\n")
chisq.test(tab)

## 4. एकाधिक रैखिक प्रतिगमन (Multiple Linear Regression)

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. प्रतिगमन निदान (Regression Diagnostics)

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. विश्वास अंतराल (Confidence Intervals)

In [ ]:
ci <- confint(model, level = 0.95)
cat("95% विश्वास अंतराल:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "अनुमान", ylab = "",
     main = "गुणांकों के लिए 95% विश्वास अंतराल")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. वन-वे एनोवा (One-Way ANOVA)

क्या सिलेंडर गणनाओं के बीच माइलेज (MPG) में महत्वपूर्ण अंतर है?

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nTukey HSD पोस्ट-हॉक तुलना:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "सिलेंडर गणना के अनुसार MPG",
        xlab = "सिलेंडर", ylab = "मील प्रति गैलन",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## सारांश

- **वेल्च टी-परीक्षण**: असमायोजित एकतरफा तुलना में, मैनुअल कारों का औसत MPG अधिक होता है
- **काई-स्क्वायर**: आकस्मिकता तालिका एक जुड़ाव का सुझाव देती है लेकिन कम अपेक्षित गणनाएं एक अनुमान चेतावनी ट्रिगर करती हैं, इसलिए इस परिणाम की सावधानीपूर्वक व्याख्या करें
- **प्रतिगमन**: समायोजन के बाद वजन और हॉर्सपावर महत्वपूर्ण नकारात्मक भविष्यवक्ता हैं; ट्रांसमिशन प्रकार इस मॉडल में महत्वपूर्ण नहीं है
- **एनोवा**: 4, 6, और 8-सिलेंडर समूहों के बीच MPG महत्वपूर्ण रूप से भिन्न होता है; टुकी परिणाम युग्मित अंतरों की पहचान करते हैं